# SFT Data Preparation — Phi-3 Template

Pipeline completo de preparação de dados para Supervised Fine-Tuning (SFT):

1. Carrega o dataset de pares gerados (JSON)
2. Formata no **Phi-3 prompt style template**
3. Tokeniza com GPT-2 tokenizer (BPE, vocab 50.257)
4. Adiciona padding token `<|endoftext|>` (ID 50256)
5. Cria vetores `input_ids` e `target_ids` (deslocados em 1)
6. Mascara instruções no target com `-100`
7. Exporta `InstructionDataset` (PyTorch) + DataLoaders prontos para treino

> **Referência:** slide 07: Fine-tuning para Seguir Instruções (Prof. Raimundo Moura, DC/UFPI)

## 0. Instalação de dependências

In [6]:
!uv add transformers datasets tiktoken torch

Resolved 175 packages in 3ms
Audited 155 packages in 1.06s


## 1. Imports e configurações globais

In [7]:
import os
import sys
import torch

print(f"python_executable: {sys.executable}")
print(f"torch_version    : {torch.__version__}")
print(f"cuda_build       : {torch.version.cuda}")
print(f"is_available     : {torch.cuda.is_available()}")
print(f"device_count     : {torch.cuda.device_count()}")
print(f"cuda_visible     : {os.getenv('CUDA_VISIBLE_DEVICES')}")


python_executable: c:\Users\raffa\Documents\UFPI\Tópicos em IA\RAG\.venv\Scripts\python.exe
torch_version    : 2.11.0+cu128
cuda_build       : 12.8
is_available     : True
device_count     : 1
cuda_visible     : None


In [8]:
import json
import random
import textwrap
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer

In [9]:
# definição da seed para reprodutibilidade
SEED = 13
random.seed(SEED)
torch.manual_seed(SEED)

# parâmetros globais
# arquivo JSON gerado pelo generate_sft_pairs.py
DATASET_PATH      = "sft_dataset_docentesDC.json"

TOKENIZER_NAME    = "gpt2"          # tokenizer BPE, vocab 50.257
PAD_TOKEN_ID      = 50256           # <|endoftext|> usado como padding
IGNORE_INDEX      = -100            # valor que cross_entropy ignora
ALLOWED_MAX_LEN   = 512             # tokens máximos por amostra

TRAIN_RATIO       = 0.85 # (850 pares de treino)
VAL_RATIO         = 0.10 # (100 pares de validação)
TEST_RATIO        = 0.05 # (50 pares de teste)

BATCH_SIZE        = 8               
NUM_WORKERS       = 0

print("✅ Imports OK")
print(f"   device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

✅ Imports OK
   device: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   device: {device}")

   device: cuda


## 2. Carregamento do dataset JSON

In [5]:
def load_pairs(path: str) -> list[dict]:
    """
    Carrega o JSON de pares gerados pelo script generate_sft_pairs.py.
    Valida campos obrigatórios: instruction + output.
    """
    raw = json.loads(Path(path).read_text(encoding="utf-8"))
    # 
    pairs = raw if isinstance(raw, list) else raw.get("data", raw)

    valid = []
    for p in pairs:
        if isinstance(p, dict) and p.get("instruction") and p.get("output"):
            valid.append({
                "instruction": p["instruction"].strip(),
                "input":       p.get("input", "").strip(),
                "output":      p["output"].strip(),
            })

    print(f"📂 Dataset carregado: {len(valid)} pares válidos de {len(pairs)} totais")
    return valid


pairs = load_pairs(DATASET_PATH)

# inspeção de 2 amostras aleatórias do dataset
print("\n# Amostras #──────────────────────────────────#")
for s in random.sample(pairs, min(2, len(pairs))):
    print(f"  instruction : {s['instruction'][:90]}")
    print(f"  input       : {s['input'][:60] or '(vazio)'}")
    print(f"  output      : {s['output'][:90]}...")
    print()

📂 Dataset carregado: 1000 pares válidos de 1000 totais

# Amostras #──────────────────────────────────#
  instruction : O que são containers sequenciais em Ciência da Computação?
  input       : (vazio)
  output      : Containers sequenciais, também conhecidos como sequence containers, são estruturas de dado...

  instruction : Qual a diferença entre pilha e fila em estruturas de dados?
  input       : (vazio)
  output      : Uma pilha é uma estrutura de dados LIFO (Last In, First Out), onde o último elemento inser...



## 3. Phi-3 prompt style template

```
<|user|>
{instruction}
{input}
<|assistant|>
{output}<|endoftext|>
```

O template Phi-3 usa tokens especiais de papel (`<|user|>`, `<|assistant|>`) em vez dos delimitadores Markdown (`### Instruction:`) do Alpaca. Isso é mais próximo do padrão usado por modelos de chat modernos.

In [ ]:
#  Tokens especiais do Phi-3
USER_TOKEN      = "<|user|>"
ASSISTANT_TOKEN = "<|assistant|>"
END_TOKEN       = "<|endoftext|>"   # token 50256 no vocab GPT-2


def format_phi3(pair: dict) -> tuple[str, str]:
    """
    Formata um par no Phi-3 template.

    Retorna:
        prompt_only  - parte de instrução (que será mascarada no target)
        full_text    - texto completo (instruction + response)

    Exemplo de full_text:
        <|user|>
        O que é uma árvore AVL?
        <|assistant|>
        Uma árvore AVL é...<|endoftext|>
    """
    user_part = pair["instruction"]
    if pair["input"]:
        user_part = f"{pair['instruction']}\n{pair['input']}"

    prompt_only = f"{USER_TOKEN}\n{user_part}\n{ASSISTANT_TOKEN}\n"
    full_text   = f"{prompt_only}{pair['output']}{END_TOKEN}"

    return prompt_only, full_text


# demonstração visual
sample_pair = pairs[0]
prompt_demo, full_demo = format_phi3(sample_pair)

print("# Phi-3 template (exemplo) ─────────────────────")
print("[PROMPT ONLY: será mascarado no target]")
print(repr(prompt_demo))
print()
print("[FULL TEXT: input do modelo]")
print(full_demo[:300] + ("..." if len(full_demo) > 300 else ""))

# Phi-3 template (exemplo) ─────────────────────
[PROMPT ONLY: será mascarado no target]
'<|user|>\nO que é algoritmo minimax?\n<|assistant|>\n'

[FULL TEXT: input do modelo]
<|user|>
O que é algoritmo minimax?
<|assistant|>
Algoritmo minimax é um método recursivo utilizado em jogos de estratégia com dois jogadores, onde cada jogador tenta maximizar (MAX) ou minimizar (MIN) o resultado. A árvore de jogo é percorrida para encontrar a melhor jogada possível, com MAX escolh...


## 4. Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

# GPT-2 não tem padding token nativo; usamos <|endoftext|>
tokenizer.pad_token    = tokenizer.eos_token   # ID 50256
tokenizer.pad_token_id = PAD_TOKEN_ID

# adicionammos os tokens especiais do Phi-3 ao vocabulário
special_tokens = {"additional_special_tokens": [USER_TOKEN, ASSISTANT_TOKEN]}
tokenizer.add_special_tokens(special_tokens)

print(f"✅ Tokenizer: {TOKENIZER_NAME}")
print(f"   vocab size       : {tokenizer.vocab_size}")
print(f"   pad_token_id     : {tokenizer.pad_token_id}  ({tokenizer.pad_token!r})")
print(f"   eos_token_id     : {tokenizer.eos_token_id}")

USER_TOKEN_ID      = tokenizer.convert_tokens_to_ids(USER_TOKEN)
ASSISTANT_TOKEN_ID = tokenizer.convert_tokens_to_ids(ASSISTANT_TOKEN)
print(f"   <|user|> id      : {USER_TOKEN_ID}")
print(f"   <|assistant|> id : {ASSISTANT_TOKEN_ID}")

✅ Tokenizer: gpt2
   vocab size       : 50257
   pad_token_id     : 50256  ('<|endoftext|>')
   eos_token_id     : 50256
   <|user|> id      : 50257
   <|assistant|> id : 50258


## 5. Pré-processamento: tokenize → padding → target → máscara

Cada amostra passa por 5 sub-etapas:

| Etapa | O que faz |
|---|---|
| 2.1 | Formata no template Phi-3 |
| 2.2 | Tokeniza `full_text` → `input_ids` |
| 2.3 | Adiciona `50256` (padding) até `ALLOWED_MAX_LEN` |
| 2.4 | `target_ids` = `input_ids` deslocados 1 posição à direita |
| 2.5 | Substitui tokens da instrução por `-100` no `target_ids` |

In [9]:
def find_assistant_start(input_ids: list[int]) -> int:
    """
    Localiza o índice do primeiro token APÓS <|assistant|>\n no input_ids.
    Tudo antes desse índice (inclusive) será mascarado com IGNORE_INDEX.
    Retorna len(input_ids) se não encontrado (mascara tudo, amostra descartável).
    """
    assistant_id = ASSISTANT_TOKEN_ID
    for i, tok in enumerate(input_ids):
        if tok == assistant_id:
            # Pula o \n que segue o token <|assistant|>
            return i + 2   # +1 para o próprio token, +1 para o \n
    return len(input_ids)


def preprocess_pair(pair: dict) -> Optional[dict]:
    """
    Pipeline completo de pré-processamento de um par.

    Retorna dict com:
        input_ids  : tensor [seq_len]  — tokens da entrada completa
        target_ids : tensor [seq_len]  — tokens alvo (deslocados + mascarados)
        attention_mask : tensor [seq_len]  — 1 para tokens reais, 0 para padding

    Retorna None se a sequência exceder ALLOWED_MAX_LEN.
    """
    # 2.1 - formatamos no template Phi-3
    prompt_only, full_text = format_phi3(pair)

    # 2.2 - Tokenizar
    full_ids   = tokenizer.encode(full_text,   add_special_tokens=False)
    prompt_ids = tokenizer.encode(prompt_only, add_special_tokens=False)

    # descartamos amostras muito longas
    if len(full_ids) > ALLOWED_MAX_LEN:
        return None

    # 2.3 - Padding com 50256 até ALLOWED_MAX_LEN
    n_pad = ALLOWED_MAX_LEN - len(full_ids)
    input_ids      = full_ids + [PAD_TOKEN_ID] * n_pad
    attention_mask = [1] * len(full_ids) + [0] * n_pad

    # 2.4 - Criar target_ids deslocados 1 token à direita
    #   input :  [t0, t1, t2, ..., tN, PAD, PAD]
    #   target:  [t1, t2, t3, ..., tN, PAD, PAD]  + 1 PAD extra no fim
    target_ids = input_ids[1:] + [PAD_TOKEN_ID]

    # 2.5 - Mascarar a instrução no target (-100)
    #   localizamos onde termina o prompt (até <|assistant|>\n)
    response_start = find_assistant_start(full_ids)

    # mascaramos tudo antes da resposta (instrução + delimitadores)
    target_ids[:response_start] = [IGNORE_INDEX] * response_start

    # mascaramos tokens de padding no target (exceto o 1º, para o modelo
    # aprender a gerar o end-of-text)
    first_pad_in_target = None
    for i in range(response_start, len(target_ids)):
        if target_ids[i] == PAD_TOKEN_ID:
            if first_pad_in_target is None:
                first_pad_in_target = i   # mantém o 1º PAD
            else:
                target_ids[i] = IGNORE_INDEX   # mascara os demais

    return {
        "input_ids":       torch.tensor(input_ids,      dtype=torch.long),
        "target_ids":      torch.tensor(target_ids,     dtype=torch.long),
        "attention_mask":  torch.tensor(attention_mask, dtype=torch.long),
        "response_start":  response_start,
        "original":        pair,
    }


# Teste unitário
print("# Teste de pré-processamento #─────────────────#")
sample = preprocess_pair(pairs[0])
iids = sample["input_ids"] # type: ignore
tids = sample["target_ids"] # type: ignore
mask = sample["attention_mask"] # type: ignore
rs   = sample["response_start"] # type: ignore

print(f"  input_ids  shape : {iids.shape}")
print(f"  target_ids shape : {tids.shape}")
print(f"  response_start   : {rs} (tokens da instrução mascarados)")
print(f"  tokens reais     : {mask.sum().item()} / {len(iids)}")
print()
print(f"  input_ids  [:10]  : {iids[:10].tolist()}")
print(f"  target_ids [:10]  : {tids[:10].tolist()}  ← -100 = mascarado")
print(f"  target_ids [rs:rs+5]: {tids[rs:rs+5].tolist()}  ← resposta real")

assert (tids[:rs] == IGNORE_INDEX).all(), "Instrução deveria estar toda mascarada!"
assert tids[rs].item() != IGNORE_INDEX,   "Início da resposta não deveria ser -100!"
print("\n✅ Asserções passaram")

# Teste de pré-processamento #─────────────────#
  input_ids  shape : torch.Size([512])
  target_ids shape : torch.Size([512])
  response_start   : 15 (tokens da instrução mascarados)
  tokens reais     : 129 / 512

  input_ids  [:10]  : [50257, 198, 46, 8358, 38251, 435, 7053, 270, 5908, 10356]
  target_ids [:10]  : [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100]  ← -100 = mascarado
  target_ids [rs:rs+5]: [7053, 270, 5908, 10356, 897]  ← resposta real

✅ Asserções passaram


### Visualização das máscaras

In [10]:
def show_masking(processed: dict, n_tokens: int = 30) -> None:
    """
    Exibe os primeiros N tokens comparando input vs target.
    Deixa claro onde começa a resposta e onde há máscara.
    """
    iids = processed["input_ids"].tolist()
    tids = processed["target_ids"].tolist()
    rs   = processed["response_start"]

    print(f"{'IDX':>4}  {'INPUT_ID':>8}  {'TARGET_ID':>10}  TOKEN")
    print("-" * 56)
    for i in range(min(n_tokens, len(iids))):
        tok_str = tokenizer.decode([iids[i]]).replace("\n", "\\n")
        region  = "[INSTR]" if i < rs else "[RESP] "
        masked  = "  ← masked" if tids[i] == IGNORE_INDEX else ""
        print(f"{i:>4}  {iids[i]:>8}  {str(tids[i]):>10}  {region}  {tok_str!r:20}{masked}")

show_masking(sample) # type: ignore

 IDX  INPUT_ID   TARGET_ID  TOKEN
--------------------------------------------------------
   0     50257        -100  [INSTR]  '<|user|>'            ← masked
   1       198        -100  [INSTR]  '\\n'                 ← masked
   2        46        -100  [INSTR]  'O'                   ← masked
   3      8358        -100  [INSTR]  ' que'                ← masked
   4     38251        -100  [INSTR]  ' é'                  ← masked
   5       435        -100  [INSTR]  ' al'                 ← masked
   6      7053        -100  [INSTR]  'gor'                 ← masked
   7       270        -100  [INSTR]  'it'                  ← masked
   8      5908        -100  [INSTR]  'mo'                  ← masked
   9     10356        -100  [INSTR]  ' minim'              ← masked
  10       897        -100  [INSTR]  'ax'                  ← masked
  11        30        -100  [INSTR]  '?'                   ← masked
  12       198        -100  [INSTR]  '\\n'                 ← masked
  13     50258        -10

## 6. Processar todo o dataset

In [11]:
from tqdm.auto import tqdm

processed_all = []
skipped = 0

for p in tqdm(pairs, desc="Pré-processando"):
    result = preprocess_pair(p)
    if result is not None:
        processed_all.append(result)
    else:
        skipped += 1

print(f"\n✅ Processados : {len(processed_all)}")
print(f"   Descartados (> {ALLOWED_MAX_LEN} tokens): {skipped}")

# Estatísticas de comprimento
lengths = [s["attention_mask"].sum().item() for s in processed_all]
print(f"# Comprimento dos tokens reais (sem padding) #")
print(f"   min  : {min(lengths)}")
print(f"   max  : {max(lengths)}")
print(f"   média: {sum(lengths)/len(lengths):.1f}")

Pré-processando: 100%|██████████| 1000/1000 [00:00<00:00, 3318.05it/s]


✅ Processados : 1000
   Descartados (> 512 tokens): 0
# Comprimento dos tokens reais (sem padding) #
   min  : 58
   max  : 358
   média: 154.0


In [12]:
print("\n# Amostra processada (exemplo) #───────────────────────#")
sample = processed_all[0]
print(f"{sample}")


# Amostra processada (exemplo) #───────────────────────#
{'input_ids': tensor([50257,   198,    46,  8358, 38251,   435,  7053,   270,  5908, 10356,
          897,    30,   198, 50258,   198,  2348,  7053,   270,  5908, 10356,
          897, 38251, 23781,   285, 25125, 24313,   664,  1834, 23593,  7736,
          528,  4533,   795, 48342,   418,   390,  1556, 10366,  2634,    70,
          544,   401,   466,   271, 48342,   324,  2850,    11,   319,  2934,
          269,  4763, 48342,  7079, 11105,    64, 12991,   528,   283,   357,
        22921,     8,   267,    84, 10356,   528,   283,   357, 23678,     8,
          267,  1255,  4533,    13,   317,  6184,    94,    81,    85,   382,
          390,   474, 24076, 38251,   583, 10215,    81,  3755, 31215,  2207,
          756, 20040,   257,  7758, 17899, 48342,  4763,  1184,  8836,   626,
           11,   401, 25882,  3671,   349, 15631,    78,   267, 17266,  1504,
         1188,   273, 23430,  1226,    71,   418,   304, 20625,  3671,

## 7. Split treino / validação / teste

In [13]:
random.shuffle(processed_all)

n      = len(processed_all)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * VAL_RATIO)

train_data = processed_all[:n_train]
val_data   = processed_all[n_train : n_train + n_val]
test_data  = processed_all[n_train + n_val :]

print(f"Split:")
print(f"  treino    : {len(train_data)} ({len(train_data)/n*100:.1f}%)")
print(f"  validação : {len(val_data)}   ({len(val_data)/n*100:.1f}%)")
print(f"  teste     : {len(test_data)}   ({len(test_data)/n*100:.1f}%)")

Split:
  treino    : 850 (85.0%)
  validação : 100   (10.0%)
  teste     : 50   (5.0%)


## 8. PyTorch Dataset e DataLoaders

In [14]:
class InstructionDataset(Dataset):
    """
    Dataset PyTorch para fine-tuning com instrução mascarada.

    Cada item retorna:
        input_ids      : LongTensor [ALLOWED_MAX_LEN]
        target_ids     : LongTensor [ALLOWED_MAX_LEN]  (com -100 nos tokens mascarados)
        attention_mask : LongTensor [ALLOWED_MAX_LEN]
    """
    def __init__(self, data: list[dict]):
        self.data = data

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> dict:
        sample = self.data[idx]
        return {
            "input_ids":      sample["input_ids"],
            "target_ids":     sample["target_ids"],
            "attention_mask": sample["attention_mask"],
        }


def collate_fn(batch: list[dict]) -> dict:
    """
    Collate para batches de tamanho VARIÁVEL.

    Usa o comprimento máximo REAL dentro do batch (não ALLOWED_MAX_LEN)
    para economizar memória quando os exemplos são curtos.
    Preenche com PAD_TOKEN_ID (input) e IGNORE_INDEX (target).
    """
    max_len = max(s["attention_mask"].sum().item() for s in batch)

    input_ids_list      = []
    target_ids_list     = []
    attention_mask_list = []

    for s in batch:
        real_len = s["attention_mask"].sum().item()
        pad_len  = max_len - real_len

        input_ids_list.append(
            torch.cat([s["input_ids"][:real_len],
                       torch.full((pad_len,), PAD_TOKEN_ID, dtype=torch.long)])
        )
        target_ids_list.append(
            torch.cat([s["target_ids"][:real_len],
                       torch.full((pad_len,), IGNORE_INDEX, dtype=torch.long)])
        )
        attention_mask_list.append(
            torch.cat([s["attention_mask"][:real_len],
                       torch.zeros(pad_len, dtype=torch.long)])
        )

    return {
        "input_ids":      torch.stack(input_ids_list),
        "target_ids":     torch.stack(target_ids_list),
        "attention_mask": torch.stack(attention_mask_list),
    }


# # Instanciação dos datasets ───────────────────────────────────────────────────────
train_dataset = InstructionDataset(train_data)
val_dataset   = InstructionDataset(val_data)
test_dataset  = InstructionDataset(test_data)

# # DataLoaders ───────────────────────────────────────────────────────────────
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print(f"✅ Datasets e DataLoaders prontos")
print(f"   train_loader : {len(train_loader)} batches de até {BATCH_SIZE} amostras")
print(f"   val_loader   : {len(val_loader)} batches")
print(f"   test_loader  : {len(test_loader)} batches")

✅ Datasets e DataLoaders prontos
   train_loader : 107 batches de até 8 amostras
   val_loader   : 13 batches
   test_loader  : 7 batches


## 9. Inspeção do primeiro batch

In [15]:
batch = next(iter(train_loader))

print("── Primeiro batch do train_loader ───────────────")
print(f"  input_ids      shape : {batch['input_ids'].shape}")
print(f"  target_ids     shape : {batch['target_ids'].shape}")
print(f"  attention_mask shape : {batch['attention_mask'].shape}")
print()

# Verifica que cross_entropy vai ignorar os -100 corretamente
target = batch["target_ids"]
n_masked = (target == IGNORE_INDEX).sum().item()
n_active = (target != IGNORE_INDEX).sum().item()
print(f"  tokens com -100 (mascarados)  : {n_masked}")
print(f"  tokens ativos (perda calculada): {n_active}")
print(f"  % de tokens que contribuem para a perda: {n_active/(n_masked+n_active)*100:.1f}%")
print()

# Decodifica a 1ª amostra do batch para inspeção
print("── Decodificação da 1ª amostra ──────────────────")
first_input  = batch["input_ids"][0]
first_target = batch["target_ids"][0]
real_len = (batch["attention_mask"][0] == 1).sum().item()

print("[INPUT — primeiros 60 tokens reais]")
print(tokenizer.decode(first_input[:60].tolist()))
print()
print("[TARGET — tokens ativos (sem -100), primeiros 60]")
active_tokens = [t for t in first_target[:real_len].tolist() if t != IGNORE_INDEX]
print(tokenizer.decode(active_tokens[:60]))

── Primeiro batch do train_loader ───────────────
  input_ids      shape : torch.Size([8, 190])
  target_ids     shape : torch.Size([8, 190])
  attention_mask shape : torch.Size([8, 190])

  tokens com -100 (mascarados)  : 563
  tokens ativos (perda calculada): 957
  % de tokens que contribuem para a perda: 63.0%

── Decodificação da 1ª amostra ──────────────────
[INPUT — primeiros 60 tokens reais]
<|user|>
Qual é a principal diferença entre WGS e outros tipos de sequenciamentos mencionados?
<|assistant|>
WGS, ou Sequenciamento do Genoma Completo, se diferencia dos outros tip

[TARGET — tokens ativos (sem -100), primeiros 60]
GS, ou Sequenciamento do Genoma Completo, se diferencia dos outros tipos de sequenciamentos, como o Exoma Completo e os Paineis com Alvos Específicos, pois capt


## 10. Salvar artefatos para reutilização

In [18]:
import os

SAVE_DIR = Path("sft_artifacts")
SAVE_DIR.mkdir(exist_ok=True)

# Salva os tensores processados (sem precisar re-tokenizar)
torch.save({
    "train": train_data,
    "val":   val_data,
    "test":  test_data,
    "config": {
        "tokenizer":        TOKENIZER_NAME,
        "allowed_max_len":  ALLOWED_MAX_LEN,
        "pad_token_id":     PAD_TOKEN_ID,
        "ignore_index":     IGNORE_INDEX,
        "user_token_id":    USER_TOKEN_ID,
        "assistant_token_id": ASSISTANT_TOKEN_ID,
        "batch_size":       BATCH_SIZE,
        "seed":             SEED,
    }
}, SAVE_DIR / "processed_dataset.pt")

# Salva o tokenizer com os tokens especiais adicionados
tokenizer.save_pretrained(SAVE_DIR / "tokenizer")

# Resumo em JSON para referência rápida
summary = {
    "total_pairs":     len(processed_all),
    "train":           len(train_data),
    "val":             len(val_data),
    "test":            len(test_data),
    "skipped":         skipped,
    "allowed_max_len": ALLOWED_MAX_LEN,
    "avg_real_tokens": round(sum(lengths) / len(lengths), 1),
    "template":        "phi3",
}
(SAVE_DIR / "summary.json").write_text(json.dumps(summary, indent=2))

print(f"✅ Artefatos salvos em '{SAVE_DIR}/'")
print(f"   processed_dataset.pt  — tensores prontos para treino")
print(f"   tokenizer/            — tokenizer com tokens especiais")
print(f"   summary.json          — resumo do dataset")
print()
print(json.dumps(summary, indent=2))

✅ Artefatos salvos em 'sft_artifacts/'
   processed_dataset.pt  — tensores prontos para treino
   tokenizer/            — tokenizer com tokens especiais
   summary.json          — resumo do dataset

{
  "total_pairs": 1000,
  "train": 850,
  "val": 100,
  "test": 50,
  "skipped": 0,
  "allowed_max_len": 512,
  "avg_real_tokens": 154.0,
  "template": "phi3"
}


## 11. Como carregar e usar nos próximos notebooks

In [19]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from pathlib import Path

SAVE_DIR     = Path("sft_artifacts")
IGNORE_INDEX = -100
PAD_TOKEN_ID = 50256

# Carrega artefatos
checkpoint   = torch.load(SAVE_DIR / "processed_dataset.pt")
train_data   = checkpoint["train"]
val_data     = checkpoint["val"]
test_data    = checkpoint["test"]
cfg          = checkpoint["config"]
tokenizer    = AutoTokenizer.from_pretrained(SAVE_DIR / "tokenizer")

# Re-instancia Dataset e DataLoader (mesma classe definida acima)
train_loader = DataLoader(
    InstructionDataset(train_data),
    batch_size=cfg["batch_size"],
    shuffle=True,
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    InstructionDataset(val_data),
    batch_size=cfg["batch_size"],
    shuffle=False,
    collate_fn=collate_fn,
)

---
## Resumo do pipeline

```
JSON (pares brutos)
    │
    ▼  format_phi3()
  <|user|>\nInstrução\n<|assistant|>\nResposta<|endoftext|>
    │
    ▼  tokenizer.encode()
  input_ids: [21106, 318, 281, ..., 50256]
    │
    ▼  padding com 50256 até ALLOWED_MAX_LEN
  input_ids: [21106, ..., 50256, 50256, 50256]
    │
    ▼  deslocamento de 1 token
  target_ids: [318, 281, ..., 50256, 50256, 50256]
    │
    ▼  mascarar instrução + padding extra
  target_ids: [-100, -100, ..., T_resp, ..., 50256, -100, -100]
    │                              ▲
    │                    apenas a resposta contribui
    │                    para a perda (cross-entropy)
    ▼
  InstructionDataset → DataLoader (batches variáveis)
```